# GuardianDrive `sensing_gen`: quickstart

Three components from the project, in about two minutes on a free CPU runtime:

1. **Generate** 10-s single-lead ECG with the conditional diffusion model (trained on PTB-XL lead II).
2. **Corrupt** it with the physics sensor model (motion, impedance change, mains, EMG, contact loss), the augmentation that lifted AUROC under real 0 dB noise from 0.654 to 0.818.
3. **Query** the neural surrogate of the finite-element electrode solver (0.02 % error) and check it against the solver itself.

Results, protocol and limitations: [README](https://github.com/AKilalours/guardian-drive) · [model card](https://github.com/AKilalours/guardian-drive/blob/main/docs/MODEL_CARD.md) · [interactive explorer](https://akilalours.github.io/guardian-drive/docs/explorer.html)

In [ ]:
!pip -q install "git+https://github.com/AKilalours/guardian-drive.git"

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sensing_gen import api

gen = api.ECGGenerator.from_pretrained()          # weights ship with the package
x_norm, _ = gen.sample(4, label="norm", hr_bpm=60, seed=0)
x_abn, _ = gen.sample(4, label="abnormal", hr_bpm=95, seed=1)
t = np.arange(1000) / 100
fig, ax = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
ax[0].plot(t, x_norm[0], lw=0.9); ax[0].set_title("generated: NORM, requested 60 bpm"); ax[0].set_ylabel("mV")
ax[1].plot(t, x_abn[0], lw=0.9, color="C3"); ax[1].set_title("generated: abnormal, requested 95 bpm"); ax[1].set_ylabel("mV")
ax[1].set_xlabel("time [s]"); plt.tight_layout()

In [ ]:
aug = api.SensorAugmenter(seed=3, snr_db=(0, 3))   # a harsh draw: artifacts as strong as the ECG (0-3 dB SNR)
noisy, parts = aug(x_norm, return_parts=True)
fig, ax = plt.subplots(len(parts) + 1, 1, figsize=(12, 2 + 1.3 * len(parts)), sharex=True)
ax[0].plot(t, x_norm[0], lw=0.9, label="clean (generated)")
ax[0].plot(t, noisy[0], lw=0.8, alpha=0.8, label="after the physics sensor model")
ax[0].set_ylabel("mV"); ax[0].legend(loc="upper right", fontsize=8)
names = {"motion": "motion: skin-stretch potential + impedance-modulated gain", "emg": "muscle (EMG) bursts",
         "mains": "mains pickup from electrode impedance mismatch (60 Hz aliased to 40 Hz)",
         "baseline": "respiration baseline wander"}
for a, (k, v) in zip(ax[1:], sorted(parts.items())):
    a.plot(t, v[0], lw=0.7, color="C2"); a.set_title(names.get(k, k), fontsize=9, loc="left"); a.set_ylabel("mV")
ax[-1].set_xlabel("time [s]"); plt.tight_layout()
print("training uses SNR drawn from -3 to 18 dB; this cell fixes it to 0-3 dB so the artifacts are easy to see")


In [ ]:
sur = api.ElectrodeSurrogate.from_pretrained()
f = api.ElectrodeSurrogate.frequencies()
fig, ax = plt.subplots(figsize=(7, 4))
for lift in (0.0, 0.2, 0.4):
    z, sd = sur.impedance(lift=lift)
    ax.loglog(f, abs(z), label=f"surrogate, lift-off {lift:.0%}")
z_fe, _ = sur.impedance(lift=0.2, exact=True)    # the finite-element solver itself (~0.3 s)
ax.loglog(f, abs(z_fe), "k.", label="FE solver, lift-off 20%")
ax.set_xlabel("frequency [Hz]"); ax.set_ylabel("|Z| [ohm]"); ax.legend(); plt.tight_layout()
z_s, _ = sur.impedance(lift=0.2)
print(f"max |Z| difference surrogate vs FE: {100 * np.max(abs(abs(z_s) / abs(z_fe) - 1)):.3f} %")

Try your own: `sur.impedance(w_el_mm=6, sig_sc=8e-6, t_fat_mm=9)`, or the command line: `!sensing-gen impedance --w-el-mm 6 --lift 0.3`.
The live in-browser version is the **Electrode designer** tab of the [explorer](https://akilalours.github.io/guardian-drive/docs/explorer.html).